# Day 2 - Toolbox & Evaluation - SOLUTIONS

> Instructor copy. Every TODO is filled in and every question answered.
> The student copy is the same notebook with these cells blanked.

## Setup

Same `coursekit` imports, plus `statsforecast` for the models and
`utilsforecast` for the metrics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import (HistoricAverage, Naive, RandomWalkWithDrift,
                                  SeasonalNaive)
from statsmodels.stats.diagnostic import acorr_ljungbox
from utilsforecast.losses import mae, mape, mase, rmse, rmsse

from coursekit import checks
from coursekit import datasets as D
from coursekit import leaderboard as lb
from coursekit import plotting as P

P.use_course_style()

spine = D.spine()
H = 24
train, test = D.train_test(spine, h=H)
print(f"train: {len(train)} months to {train['ds'].max().date()}")
print(f"test : {len(test)} months from {test['ds'].min().date()}")

---
# Exercise 2.1 - The benchmark floor

*Follows segment 1. 13 minutes.*

Fit all four benchmarks and look at them. Everything for the rest of the course
is measured against these.

In [ ]:
MODELS = ["HistoricAverage", "Naive", "SeasonalNaive", "RWD"]
LABELS = {"HistoricAverage": "Mean", "Naive": "Naive",
          "SeasonalNaive": "Seasonal naive", "RWD": "Drift"}

sf = StatsForecast(
    models=[HistoricAverage(), Naive(), SeasonalNaive(season_length=12),
            RandomWalkWithDrift()],
    freq=D.FREQ, n_jobs=1,
)
fc = sf.forecast(df=train, h=H, level=[80, 95], fitted=True)

checks.check_ex_2_1(fc, MODELS)
fc.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
hist = train.tail(72)
ax.plot(hist["ds"], hist["y"], color=P.BLACK, lw=1.1, label="observed")
ax.plot(test["ds"], test["y"], color=P.GREY, lw=1.4, ls="--", label="actual")
for m, c in zip(MODELS, [P.PINK, P.GREEN, P.ORANGE, P.BLUE]):
    ax.plot(fc["ds"], fc[m], lw=1.7, color=c, label=LABELS[m])
ax.set(title="Four benchmarks, 24 months ahead")
ax.legend(frameon=False, ncols=3)
plt.show()

**Question.** Two of these are obviously wrong before you compute a single
metric. Which, and why?


*Your answer:*


*Answer.* The **mean** method forecasts a flat line at roughly 160 for a series
currently sitting near 370 - it averages over 37 years of growth, so it is
hopeless on any trending series. The **naive** method forecasts a flat line at
the last value, which throws away the seasonality we spent all of Day 1
establishing. **Drift** at least captures the trend but still ignores season.
Only the **seasonal naive** reproduces the annual shape.

### Stretch - forecasting on a transformed scale

The spine is multiplicative. Forecast the Box-Cox transformed series, then
back-transform. Note that the naive back-transform gives you the **median**, not
the mean.

In [ ]:
from coreforecast.scalers import boxcox, boxcox_lambda, inv_boxcox

lam = boxcox_lambda(train["y"].to_numpy(), method="loglik")
train_t = train.assign(y=boxcox(train["y"].to_numpy(), lam))

sf_t = StatsForecast(models=[SeasonalNaive(season_length=12)], freq=D.FREQ, n_jobs=1)
fc_t = sf_t.forecast(df=train_t, h=H, level=[80])
back = inv_boxcox(fc_t["SeasonalNaive"].to_numpy(), lam)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train.tail(48)["ds"], train.tail(48)["y"], color=P.BLACK, lw=1.1, label="observed")
ax.plot(test["ds"], test["y"], color=P.GREY, ls="--", lw=1.4, label="actual")
ax.plot(fc["ds"], fc["SeasonalNaive"], color=P.ORANGE, lw=1.6, label="SNaive, raw scale")
ax.plot(fc_t["ds"], back, color=P.BLUE, lw=1.6, label="SNaive, via Box-Cox")
ax.legend(frameon=False, ncols=2)
ax.set(title=f"Forecasting on the transformed scale (lambda = {lam:.3f})")
plt.show()

print("Back-transforming a forecast gives the MEDIAN of the forecast "
      "distribution, not the mean. For a skewed distribution those differ, and "
      "if you are adding forecasts up (across stores, across months) you need "
      "means. That correction is the 'bias adjustment' in Ch 5.6.")

---
# Exercise 2.2 - Are the residuals white noise?

*Follows segment 2. 13 minutes.*

If a model's residuals still carry structure, the model has not finished.

In [ ]:
fv = sf.forecast_fitted_values()

resid = fv["y"] - fv["SeasonalNaive"]
fig, axes = P.residual_diagnostics(resid, ds=fv["ds"],
                                   title="Seasonal naive residuals")
plt.show()

lb_pvalue = float(acorr_ljungbox(resid.dropna(), lags=[24],
                                 return_df=True)["lb_pvalue"].iloc[0])

print(f"mean residual : {pd.Series(resid).mean():.3f}")
print(f"Ljung-Box p   : {lb_pvalue:.3e}")

checks.check_ex_2_2(resid, lb_pvalue)

In [ ]:
resid_d = fv["y"] - fv["RWD"]
fig, axes = P.residual_diagnostics(resid_d, ds=fv["ds"], title="Drift residuals")
plt.show()

for name, r in [("SeasonalNaive", resid), ("RWD", resid_d)]:
    r = r.dropna()
    p = float(acorr_ljungbox(r, lags=[24], return_df=True)["lb_pvalue"].iloc[0])
    first_half, second_half = r.iloc[:len(r) // 2], r.iloc[len(r) // 2:]
    print(f"{name:<14} mean={r.mean():8.3f}  LB p={p:.2e}  "
          f"sd early={first_half.std():6.2f}  sd late={second_half.std():6.2f}")

**Write your verdict.** For each method, which of the four residual properties
hold, and what does that imply?


*Your answer:*


*Answer.* Neither is close to white noise.

- **Uncorrelated:** fails badly for both - Ljung-Box p is effectively zero and
  the residual ACF has large spikes. There is a great deal of signal left.
- **Zero mean:** the seasonal naive's mean residual is clearly positive, because
  the series trends upward and last year's value is systematically too low. That
  is a *bias*: the forecast will be low every time.
- **Constant variance:** fails - the late-period standard deviation is several
  times the early one, because the series grew eightfold. This is exactly what
  the Box-Cox transform in 1.4 addresses.
- **Normal:** roughly, but with heavy tails.

Implication: the benchmark floor is a floor, not a model. The failures are
informative - the bias says "add a trend", the seasonal spikes say "the seasonal
shape has changed", the variance says "transform first".

---
# Exercise 2.3 - Intervals, and how much to believe them

*Follows segment 3. 15 minutes.*

In [ ]:
fan = fc.rename(columns={
    "SeasonalNaive": "mean",
    "SeasonalNaive-lo-80": "lo-80", "SeasonalNaive-hi-80": "hi-80",
    "SeasonalNaive-lo-95": "lo-95", "SeasonalNaive-hi-95": "hi-95",
})
fig, ax = plt.subplots(figsize=(10, 4.6))
P.fan_chart(train, fan, levels=(80, 95), ax=ax, actual=test, history_tail=72,
            title="Seasonal naive with prediction intervals")
plt.show()

In [ ]:
width = fc["SeasonalNaive-hi-80"] - fc["SeasonalNaive-lo-80"]

h = np.arange(1, len(width) + 1)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(h, width, color=P.BLUE, lw=2, label="actual width")
ax.plot(h, width.iloc[0] * np.sqrt(h), color=P.ORANGE, ls="--", lw=1.4,
        label="width_1 * sqrt(h)")
ax.set(xlabel="horizon h", ylabel="80% interval width", title="Widening with h")
ax.legend(frameon=False)
plt.show()

In [ ]:
merged = test.merge(fc, on=["unique_id", "ds"])

coverage = float(((merged["y"] >= merged["SeasonalNaive-lo-80"])
                  & (merged["y"] <= merged["SeasonalNaive-hi-80"])).mean())
se = float(np.sqrt(0.8 * 0.2 / len(merged)))

print(f"nominal 80%,  measured {coverage:.1%}  +/- {1.96 * se:.1%} (95% CI)")
checks.check_ex_2_3(width, coverage, se)

**Question.** Your measured coverage came with an error bar roughly 16 points
wide. What would you have to change to measure coverage properly - and is that
what exercise 2.5 does?


*Your answer:*


*Answer.* You need more scored points, and they must come from *different
origins* rather than from extending one test window (extending it just forecasts
further ahead, where the model is worse). Rolling-origin cross-validation gives
exactly that: 8 folds x 12 months = 96 scored points instead of 24, cutting the
standard error in half. That is exercise 2.5.

Note it does not fix the *other* problem - the interval formula ignores model
uncertainty - so even a well-measured coverage tends to come in under nominal.

---
# Exercise 2.4 - Scoring, and the metric that lies

*Follows segment 4. 12 minutes.*

In [ ]:
scores = pd.DataFrame({
    "MAE": mae(merged, models=MODELS)[MODELS].iloc[0],
    "RMSE": rmse(merged, models=MODELS)[MODELS].iloc[0],
    "MAPE_pct": mape(merged, models=MODELS)[MODELS].iloc[0] * 100,
    "MASE": mase(merged, models=MODELS, seasonality=12, train_df=train)[MODELS].iloc[0],
    "RMSSE": rmsse(merged, models=MODELS, seasonality=12, train_df=train)[MODELS].iloc[0],
})
scores.index = [LABELS[m] for m in scores.index]

checks.check_ex_2_4(scores)
scores.round(3)

Now build the case where MAPE misleads. Construct a near-zero series and two
forecasts: one that is a little too **low**, one that is much too **high**.

In [ ]:
rng = np.random.default_rng(3)
n = 48
low = pd.DataFrame({
    "ds": pd.date_range("2020-01-01", periods=n, freq="MS"),
    "y": np.clip(rng.poisson(1.4, n).astype(float), 0.2, None),
})

pred_hi = low["y"] + 2.0
pred_lo = low["y"] - 0.15
for name, pred in [("A: +2.00 units", pred_hi), ("B: -0.15 units", pred_lo)]:
    err = low["y"] - pred
    print(f"{name:<18} MAE = {err.abs().mean():5.2f}   "
          f"MAPE = {(err / low['y']).abs().mean() * 100:8.1f}%")

print("\nMAE says B is 13x better, which matches the picture. MAPE agrees on "
      "direction here but wildly exaggerates: dividing a 2-unit error by an "
      "actual of 0.2 gives 1000%. On a series that ever touches zero, MAPE is "
      "undefined outright.")

**Rank the four benchmarks and defend the ranking.** Which metric did you use,
and why not the others?


*Your answer:*


*Answer.* Seasonal naive > Naive > Drift > Mean, on MASE.

MASE, because it is scale-free (so this ranking can be compared against other
series later), it is defined even when the series touches zero, and the
benchmark is built into it - a MASE of 1.11 immediately tells you the winner is
still slightly worse than a one-step seasonal naive.

Not MAE or RMSE: correct here, but their units are millions of dollars, so they
cannot be pooled across series. Not MAPE: this series never approaches zero so
it happens to behave, but selecting on MAPE builds a habit that breaks the first
time you meet slow-moving demand.

---
# Exercise 2.5 - The harness

*Follows segment 5. 17 minutes.*

This is the exercise the rest of the course rests on. You are building the
evaluation harness that every Day 3 model gets plugged into.

In [ ]:
cv = sf.cross_validation(df=spine, h=12, step_size=12, n_windows=8, level=[80])

print(f"folds : {cv['cutoff'].nunique()}")
print(f"scored points : {len(cv)}")
cv.head()

In [ ]:
rows = []
for m in MODELS:
    fold_mase, fold_rmsse = [], []
    for cut, g in cv.groupby("cutoff"):
        tr = spine[spine["ds"] <= cut]
        g1 = g.drop(columns=["cutoff"])
        fold_mase.append(mase(g1, models=[m], seasonality=12, train_df=tr)[m].iloc[0])
        fold_rmsse.append(rmsse(g1, models=[m], seasonality=12, train_df=tr)[m].iloc[0])
    inside = ((cv["y"] >= cv[f"{m}-lo-80"]) & (cv["y"] <= cv[f"{m}-hi-80"])).mean()
    rows.append({"model": LABELS[m], "mase": np.mean(fold_mase),
                 "rmsse": np.mean(fold_rmsse), "coverage_80": float(inside),
                 "mase_min": np.min(fold_mase), "mase_max": np.max(fold_mase)})

summary = pd.DataFrame(rows)

checks.check_ex_2_5(cv, summary)
summary.round(3)

Write the results to the leaderboard. **This file is the course's running
scoreboard** - Day 3 appends to the same table.

In [ ]:
lb.reset()   # start clean; re-running this cell is safe

for _, row in summary.iterrows():
    lb.record(
        row["model"], day=2,
        mase=float(row["mase"]), rmsse=float(row["rmsse"]),
        coverage_80=float(row["coverage_80"]),
        notes="benchmark, 8-fold rolling origin",
    )

table = lb.show()
checks.check_leaderboard(table)
table.round(3)

### Stretch - how much does one window matter?

Score each fold separately and look at the spread.

In [ ]:
per_fold = pd.DataFrame([
    mase(g.drop(columns=["cutoff"]), models=MODELS, seasonality=12,
         train_df=spine[spine["ds"] <= cut])[MODELS].iloc[0]
    for cut, g in cv.groupby("cutoff")
]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 3.8))
for i, m in enumerate(MODELS):
    ax.scatter(np.full(len(per_fold), i), per_fold[m], s=45, color=P.ORANGE,
               alpha=0.75, zorder=3)
ax.set_xticks(range(len(MODELS)), [LABELS[m] for m in MODELS], rotation=20, ha="right")
ax.set(ylabel="MASE", title="One dot per fold")
ax.set_yscale("log")
plt.show()

sn = per_fold["SeasonalNaive"]
print(f"Seasonal naive MASE by fold: {'  '.join(f'{v:.2f}' for v in sn)}")
print(f"best {sn.min():.2f}, worst {sn.max():.2f} - a {sn.max() / sn.min():.1f}x spread")
print("\nThe RANKING was identical in every fold. The NUMBER was not. Report "
      "the ranking with confidence and the number with a spread.")

---
## End of Day 2

You have an evaluation harness: benchmarks, residual diagnostics, intervals with
an honest error bar, scale-free metrics, and rolling-origin cross-validation.

`labs/leaderboard.csv` now holds the benchmark floor. Every model on Day 3 has
to get past it.